__!!! Activate native bodies-at-rest (pressurenet) environment before running this notebook.__  
Make sure to comment/uncomment path for local or remote execution.

### Initialization

In [ ]:
# Import libraries
import os
import sys
import torch
import smplx
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from datasets import HDF5Dataset
from datetime import datetime
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

np.set_printoptions(threshold=sys.maxsize, precision=3, suppress=True)

# Paths to SMPL models & hdf5 dataset
# hdf5_file_path = 'synthetic_data/pre_processed/preprocessed_mod1_float32_add_noise_0__include_weight_height_False__omit_contact_sobel_False__use_hover_False__mod_1__normalize_per_image_True.hdf5'
hdf5_file_path = '/home/nashah/scratch/data/pre_processed/preprocessed_mod1_float32_add_noise_0__include_weight_height_False__omit_contact_sobel_False__use_hover_False__mod_1__normalize_per_image_True.hdf5'

smpl_feml_model_path_v1_0 = 'smpl/models/basicModel_f_lbs_10_207_0_v1.0.0.pkl'	# v1.0.0 has only 10 shape coefficients
smpl_male_model_path_v1_0 = 'smpl/models/basicmodel_m_lbs_10_207_0_v1.0.0.pkl'	# v1.0.0 has only 10 shape coefficients
smpl_feml_model_path_v1_1 = 'smpl/models/basicmodel_f_lbs_10_207_0_v1.1.0.pkl'	# v1.1.0 has 300 shape coefficients
smpl_male_model_path_v1_1 = 'smpl/models/basicmodel_m_lbs_10_207_0_v1.1.0.pkl'	# v1.1.0 has 300 shape coefficients
smpl_neut_model_path_v1_1 = 'smpl/models/basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl'	# neutral is only available in v1.1.0

# Load SMPL models
model = smplx.SMPL(smpl_feml_model_path_v1_0)
# model = smplx.SMPL(smpl_male_model_path_v1_0)
# model = smplx.SMPL(smpl_feml_model_path_v1_1)
# model = smplx.SMPL(smpl_male_model_path_v1_1)
# model = smplx.SMPL(smpl_neut_model_path_v1_1)

# DataLoader setup
batch_size = 1
train_dataset = HDF5Dataset(hdf5_file_path=hdf5_file_path, split='train')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

### Function

In [ ]:
def visualize_smpl_2d(body_pose, global_orient, betas, transl, out_path=None):
    # Create a unique filename using timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Create a directory named 'smpl_2d_visualizations' if it doesn't exist
    base_dir = 'smpl_2d_visualizations'
    os.makedirs(base_dir, exist_ok=True)

    if out_path is None:
        out_path = f'{base_dir}/{timestamp}_smpl_render.png'
    else:
        out_path = f'{base_dir}/{timestamp}_{out_path}.png'

    # Get output mesh
    output      = model(body_pose=body_pose, global_orient=global_orient, betas=betas, transl=transl)
    vertices    = output.vertices.detach().cpu().numpy().squeeze()
    faces       = model.faces

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Create triangle mesh
    mesh = Poly3DCollection(vertices[faces], alpha=0.9)
    mesh.set_edgecolor('k')
    ax.add_collection3d(mesh)

    # Scale to fit
    x, y, z = vertices[:, 0], vertices[:, 1], vertices[:, 2]
    max_range = np.ptp(np.concatenate([x, y, z]))
    mid_x, mid_y, mid_z = np.mean(x), np.mean(y), np.mean(z)
    ax.set_xlim(mid_x - max_range / 2, mid_x + max_range / 2)
    ax.set_ylim(mid_y - max_range / 2, mid_y + max_range / 2)
    ax.set_zlim(mid_z - max_range / 2, mid_z + max_range / 2)

    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"✅ Saved: {out_path}")

### A Random Subject from PressureNet Data

In [ ]:
for batch_index, (inputs, true_labels) in enumerate(train_loader, start=1):
	print(f"Processing Batch: {batch_index}/{len(train_loader)}")

	is_female = true_labels[:, 157].bool()

	# Extract SMPL parameters from true_labels
	betas           = true_labels[:, 72:82]     # Shape: (batch_size, 10)
	global_orient   = true_labels[:, 82:85]     # Shape: (batch_size, 3)
	body_pose       = true_labels[:, 85:154]    # Shape: (batch_size, 69)
	transl          = true_labels[:, 154:157]   # Shape: (batch_size, 3)

	# print(f"Batch Size:	{inputs.shape[0]}")
	print(f"Is Female:	{is_female}")
	print(f"Betas:		{np.array(betas)}")
	print(f"Global Orient:	{np.array(global_orient)}")
	print(f"Body Pose:	{np.array(body_pose).shape}")
	print(f"Transl:		{np.array(transl)}")

	break

visualize_smpl_2d(body_pose, global_orient, betas, transl)

### User-defined Parameters
##### 1️⃣ Full T-Pose (Arms Extended)  
👉 Good for baseline comparison

In [ ]:
# Define a custom pose (23 joints * 3D axis-angle = 69 values)
body_pose = torch.zeros(1, 69)

# Define a global orientation (pelvis rotation)
global_orient = torch.tensor([[0, 0, 0]], dtype=torch.float32)

# Define body shape parameters (identity variation)
betas = torch.tensor([[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], dtype=torch.float32)

# Define translation (position in space)
transl = torch.tensor([[0.0, 0.0, 0.0]], dtype=torch.float32)

visualize_smpl_2d(body_pose, global_orient, betas, transl)

##### 2️⃣ Right Arm Raised (Like a Wave)
👉 Shows effect of modifying specific joints

In [ ]:
body_pose = torch.tensor([
    0.0, 0.0, 0.0,  # Joint 1
    0.0, 0.0, 0.0,  # Joint 2
    0.0, 0.0, 0.0,  # Joint 3
    -1.5, 0.0, 0.0,  # Right shoulder (abduction)
    *([0] * 57)  # Keep remaining joints at default
], dtype=torch.float32).reshape(1, 69)

visualize_smpl_2d(body_pose, global_orient, betas, transl)

##### 3️⃣ Slight Forward Bend (Like Bowing)
👉 Demonstrates full-body pose change

In [ ]:
body_pose = torch.tensor([
    0.5, 0.0, 0.0,  # Leaning torso forward
    *([0] * 66)
], dtype=torch.float32).reshape(1, 69)

visualize_smpl_2d(body_pose, global_orient, betas, transl)

##### 4️⃣ Sitting Pose (Bent Knees)
👉 Good for showcasing lower body changes

In [ ]:
body_pose = torch.tensor([
    0.0, 0.0, 0.0,  # Joint 1
    0.0, 0.0, 0.0,  # Joint 2
    0.0, 0.0, 0.0,  # Joint 3
    *([0] * 24),  # Keep upper body neutral
    1.5, 0.0, 0.0,  # Right knee bent
    1.5, 0.0, 0.0,  # Left knee bent
    *([0] * 30)
], dtype=torch.float32).reshape(1, 69)

visualize_smpl_2d(body_pose, global_orient, betas, transl)

##### 5️⃣ Rotating the Entire Body (Global Orientation)
👉 Shows how the whole model rotates

In [ ]:
global_orient = torch.tensor([[0.0, 1.5, 0.0]], dtype=torch.float32)  # Rotate around Y-axis (turn left)

visualize_smpl_2d(body_pose, global_orient, betas, transl)

##### 6️⃣a. Making the Body Fat (Shape Change)
👉 Good for showing how body shapes affect visualization

In [ ]:
betas = torch.tensor([[3.0] * 10], dtype=torch.float32)  # Max positive shape params

visualize_smpl_2d(body_pose, global_orient, betas, transl)

##### 6️⃣b. Making the Body Thin (Shape Change)
👉 Good for showing how body shapes affect visualization

In [ ]:
betas = torch.tensor([[-3.0] * 10], dtype=torch.float32)  # Max negative shape params

visualize_smpl_2d(body_pose, global_orient, betas, transl)